In [53]:
#Import Modules
from typing import List, Dict
import ollama
import gradio as gr
import time

In [54]:
#Load Environment Variables


In [55]:
#Constants
OLLAMA_HOST = "http://192.168.1.239:11434"
MODEL_CHOICES = [
    ("Coding Bot", "qwen2.5:14b"),
    ("Technical Helper", "llama3:8b"),
]

In [56]:
def stream_ollama(model: str, user_message: str, history: list = None):
    if history is None:
        history = []
    messages = history + [{"role": "user", "content": user_message}]
    client = ollama.Client(host=OLLAMA_HOST)
    response_content = ""
    streamed_history = messages + [{"role": "assistant", "content": ""}]
    for chunk in client.chat(model=model, messages=messages, stream=True):
        # Append only new content
        new_text = chunk["message"]["content"]
        if new_text:
            response_content += new_text
            streamed_history[-1]["content"] = response_content
            yield streamed_history, streamed_history, ""
            time.sleep(0.03)  # 30ms delay for smoother effect


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Computacenter Chatbot")
    gr.Markdown("### Select a model and ask a question!")
    gr.Markdown("This chat bot can be used for technical or coding questions.")
    model_dropdown = gr.Dropdown(
        choices=[(name, value) for name, value in MODEL_CHOICES],
        value=MODEL_CHOICES[0][1],
        label="Select Model"
    )
    chatbox = gr.Chatbot(type="messages", render_markdown=True)
    user_input = gr.Textbox(label="Your message:")
    state = gr.State([])

    user_input.submit(
        stream_ollama,
        inputs=[model_dropdown, user_input, state],
        outputs=[chatbox, state, user_input]
    )

demo.launch()